# 07. InfraScan Road Damage Detection Training (YOLO on RDD2022) - Kaggle Edition

This notebook trains YOLO object detection models on the RDD2022 road damage dataset in the Kaggle environment.

### Manual Steps to Run on Kaggle:
1. **Create a Private Kaggle Dataset**:
   - On Kaggle, go to **Datasets > New Dataset**.
   - Upload `rdd2022_yolo_small.zip` and `ml_code_detection.zip` (located locally in `data/processed/colab_zips/`).
   - Name your dataset (e.g., `infrascan-detection` or `rdd2022-yolo-small`) and set visibility to **Private**.
2. **Attach Dataset with 'Add Input'**:
   - In this notebook, open the right-hand sidebar and click **+ Add Input**.
   - Search for your uploaded private dataset and click **Add** to attach it under `/kaggle/input/`.
3. **Set Accelerator to GPU**:
   - In the right-hand sidebar (**Notebook Settings**), select **Accelerator > GPU T4 x2** or **GPU P100**.
4. **Turn Internet ON**:
   - In the right-hand sidebar (**Notebook Settings**), toggle **Internet** to **On** (required for `pip install ultralytics` and base weights).
5. **Save Version When Finished**:
   - Once all cells finish, click **Save Version** (e.g. *Save & Run All (Commit)* or *Quick Save*) to persist `/kaggle/working/detection_results.zip` and download artifacts.

### 1. Verify Internet Connectivity and Install Ultralytics

Check that Internet access is enabled in notebook settings and install the Ultralytics YOLO package.

In [ ]:
import socket
import subprocess
import sys

# 1. Verify Internet Connectivity
def check_internet(host="8.8.8.8", port=53, timeout=3):
    try:
        socket.setdefaulttimeout(timeout)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect((host, port))
        return True
    except socket.error:
        return False

if not check_internet():
    print("[!] ERROR: Internet is turned OFF in this Kaggle session!")
    print("    Please toggle 'Internet' to ON in the right-hand Notebook Settings panel.")
    raise ConnectionError("Internet access is required to install dependencies and download YOLO weights.")
else:
    print("[OK] Internet connection verified.")

# 2. Install Ultralytics
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
import ultralytics
print(f"[OK] Ultralytics {ultralytics.__version__} installed successfully.")

### 2. Verify GPU Accelerator and Free Disk Space

Audit hardware acceleration (`device=0`) and inspect available disk space in `/kaggle/working`.

In [ ]:
import os
import shutil
import torch

# Enforce device=0 only
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if torch.cuda.is_available():
    torch.cuda.set_device(0)

print("=" * 65)
print("HARDWARE & STORAGE AUDIT (device=0)")
print("=" * 65)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name (0)    : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM Capacity   : {vram_gb:.2f} GB")
else:
    print("[!] WARNING: GPU accelerator not detected. Please select GPU in Settings.")

# Free disk space in /kaggle/working
working_dir = "/kaggle/working"
total_b, used_b, free_b = shutil.disk_usage(working_dir)
print(f"\nDisk Space in {working_dir}:")
print(f"  Total : {total_b / (1024**3):.2f} GB")
print(f"  Used  : {used_b / (1024**3):.2f} GB")
print(f"  Free  : {free_b / (1024**3):.2f} GB")
print("=" * 65)

### 3. Detect and Prepare Dataset in /kaggle/working/data

Inspect `/kaggle/input`. If the dataset was already extracted (contains `images/`, `labels/`, and `data.yaml`), copy it to `/kaggle/working/data` so Ultralytics can write its label `.cache` files. If only `.zip` archives exist, unzip them to `/kaggle/working/data`. Rewrite `data.yaml` with absolute `/kaggle/working/data` paths, print counts per split, and assert 1:1 image-to-label parity.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

input_root = Path("/kaggle/input")
working_root = Path("/kaggle/working")
working_data = working_root / "data"
working_data.mkdir(parents=True, exist_ok=True)
dest_dataset_dir = working_data / "rdd2022_yolo_small"

print("=" * 70)
print("1. LISTING /kaggle/input CONTENTS:")
print("=" * 70)
if input_root.exists():
    for root, dirs, files in os.walk(input_root):
        rel_root = Path(root).relative_to(input_root)
        print(f"  [{rel_root}] - Subdirs: {len(dirs)}, Files: {len(files)}")
        for f in files[:5]:
            print(f"    - {f}")
        if len(files) > 5:
            print(f"    ... and {len(files) - 5} more files")
else:
    print("[!] /kaggle/input does not exist!")

print("\n" + "=" * 70)
print("2. PREPARING DETECTION SOURCE CODE IN /kaggle/working:")
print("=" * 70)
code_zips = list(input_root.rglob("ml_code_detection.zip"))
code_dirs = [p for p in input_root.rglob("ml/detection") if p.is_dir()]

if (working_root / "ml" / "detection").exists():
    print(f"[OK] Detection code already present in {working_root / 'ml' / 'detection'}.")
elif code_zips:
    print(f"Extracting code archive: {code_zips[0]} to {working_root}...")
    with zipfile.ZipFile(code_zips[0], "r") as zf:
        zf.extractall(working_root)
    print("[OK] Code extracted successfully.")
elif code_dirs:
    src_ml = code_dirs[0].parent
    print(f"Copying detection code from {src_ml} to {working_root / 'ml'}...")
    shutil.copytree(src_ml, working_root / "ml", dirs_exist_ok=True)
    print("[OK] Code copied successfully.")
else:
    raise FileNotFoundError("Could not find 'ml_code_detection.zip' or 'ml/detection' in /kaggle/input!")

print("\n" + "=" * 70)
print("3. DETECTING & PREPARING DATASET IN /kaggle/working/data:")
print("=" * 70)
# If dataset is not already copied/extracted in working directory
if not (dest_dataset_dir / "images" / "train").exists():
    # Check if a folder with images/ and labels/ and data.yaml exists in /kaggle/input
    extracted_candidates = []
    for p in input_root.rglob("data.yaml"):
        p_dir = p.parent
        if (p_dir / "images").exists() and (p_dir / "labels").exists():
            extracted_candidates.append(p_dir)

    data_zips = list(input_root.rglob("rdd2022_yolo_small.zip"))

    if extracted_candidates:
        src_ds = extracted_candidates[0]
        print(f"[DETECTED] Pre-extracted dataset folder found at: {src_ds}")
        print(f"Copying to {dest_dataset_dir} (so Ultralytics can write its label .cache files)...")
        shutil.copytree(src_ds, dest_dataset_dir, dirs_exist_ok=True)
        print("[OK] Dataset copied to writable working directory.")
    elif data_zips:
        src_zip = data_zips[0]
        print(f"[DETECTED] Dataset zip archive found at: {src_zip}")
        print(f"Unzipping into {working_data}...")
        with zipfile.ZipFile(src_zip, "r") as zf:
            zf.extractall(working_data)
        print("[OK] Dataset unzipped to writable working directory.")
    else:
        raise FileNotFoundError(
            "Could not find pre-extracted dataset or 'rdd2022_yolo_small.zip' in /kaggle/input! "
            "Please ensure your dataset is attached via 'Add Input'."
        )
else:
    print(f"[OK] Dataset already present in working directory: {dest_dataset_dir}")

print("\n" + "=" * 70)
print("4. REWRITING data.yaml WITH ABSOLUTE WORKING PATHS:")
print("=" * 70)
import sys
if str(working_root) not in sys.path:
    sys.path.insert(0, str(working_root))
from ml.detection import config, train_yolo

data_yaml_path = train_yolo.make_data_yaml(
    data_root=str(dest_dataset_dir.as_posix()),
    out_path=str((dest_dataset_dir / "data.yaml").as_posix()),
)
with open(data_yaml_path, "r", encoding="utf-8") as f:
    print(f.read())

print("=" * 70)
print("5. VERIFYING IMAGE COUNTS & 1:1 IMAGE-TO-LABEL PARITY:")
print("=" * 70)
for split in ["train", "val", "test"]:
    img_split_dir = dest_dataset_dir / "images" / split
    lbl_split_dir = dest_dataset_dir / "labels" / split

    assert img_split_dir.exists(), f"Missing images directory for split: {split}"
    assert lbl_split_dir.exists(), f"Missing labels directory for split: {split}"

    img_files = sorted(list(img_split_dir.glob("*.jpg")))
    lbl_files = sorted(list(lbl_split_dir.glob("*.txt")))

    img_stems = {f.stem for f in img_files}
    lbl_stems = {f.stem for f in lbl_files}

    print(f"  Split '{split:5}': {len(img_files):5,} images | {len(lbl_files):5,} labels")

    assert len(img_files) == len(lbl_files), (
        f"Count mismatch in {split}: {len(img_files)} images vs {len(lbl_files)} labels"
    )
    assert img_stems == lbl_stems, (
        f"Parity mismatch in {split}: {len(img_stems ^ lbl_stems)} mismatched filenames!"
    )
    print(f"    -> [PASS] Every image in '{split}' has a matching label file.")

print("\n[OK] Dataset validation complete and ready in writable working directory!")

### 4. Setup Python Path and Import Detection Modules

Import the standardized InfraScan detection modules from `/kaggle/working`.

In [ ]:
import sys
from pathlib import Path

if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

from ml.detection import config, train_yolo
print("[OK] InfraScan detection modules imported successfully.")
print(f"Classes              : {config.CLASS_NAMES}")
print(f"Standard Input Size  : {config.IMAGE_SIZE}")
print(f"Standard Epochs      : {config.EPOCHS}")
print(f"Batch Size           : {config.BATCH_SIZE}")
print(f"Save Period          : {config.SAVE_PERIOD} (best.pt and last.pt only)")

### 5. Sanity Debug Run (1 Epoch, 10% Data Fraction)

Run a fast debug check on `device=0` to verify loss computation, GPU gradient flow, and run card logging without long waits.

In [ ]:
data_yaml_path = "/kaggle/working/data/rdd2022_yolo_small/data.yaml"

print("Starting Sanity Debug Run on device=0...")
debug_results = train_yolo.train(
    model_weights="yolov8s.pt",
    data_yaml=data_yaml_path,
    project_dir="/kaggle/working/debug_detection",
    run_name="debug_run",
    debug=True,
)
print("[OK] Sanity debug run completed successfully!")

### 6. Full Training Run 1: YOLOv8s Baseline (40 Epochs, Seed 42)

Execute full 40-epoch baseline training for `yolov8s.pt`.
Outputs and checkpoints persist to `/kaggle/working/models/detection/yolov8s_rdd2022/`.

In [ ]:
data_yaml_path = "/kaggle/working/data/rdd2022_yolo_small/data.yaml"
kaggle_project_dir = "/kaggle/working/models/detection"

yolov8s_results = train_yolo.train(
    model_weights="yolov8s.pt",
    data_yaml=data_yaml_path,
    project_dir=kaggle_project_dir,
    run_name="yolov8s_rdd2022",
    debug=False,
)
print("[OK] YOLOv8s full training complete!")

### 7. Full Training Run 2: YOLO11s Comparison Model (40 Epochs, Seed 42)

Execute a second training run with identical settings, seed, and data split for `yolo11s.pt`.
Outputs persist to `/kaggle/working/models/detection/yolo11s_rdd2022/`.

In [ ]:
yolo11s_results = train_yolo.train(
    model_weights="yolo11s.pt",
    data_yaml=data_yaml_path,
    project_dir=kaggle_project_dir,
    run_name="yolo11s_rdd2022",
    debug=False,
)
print("[OK] YOLO11s full training complete!")

### 8. Verify Saved Checkpoints and Run Cards

Verify that `best.pt`, `last.pt`, and `run_card.json` were created for both runs.

In [ ]:
from pathlib import Path
import json

for run_name in ["yolov8s_rdd2022", "yolo11s_rdd2022"]:
    run_folder = Path(kaggle_project_dir) / run_name
    weights_dir = run_folder / "weights"
    run_card_file = run_folder / "run_card.json"
    print(f"\n=== Checking Run: {run_name} ===")
    print(f"  Weights directory exists: {weights_dir.exists()}")
    if weights_dir.exists():
        for w in sorted(weights_dir.iterdir()):
            print(f"    + {w.name:15}: {w.stat().st_size / (1024**2):.2f} MB")
    print(f"  Run card exists: {run_card_file.exists()}")
    if run_card_file.exists():
        with open(run_card_file, "r", encoding="utf-8") as f:
            card = json.load(f)
        print(f"    Model Weights : {card.get('model_weights')}")
        print(f"    Epochs        : {card.get('hyperparameters', {}).get('epochs')}")
        print(f"    Date UTC      : {card.get('date_utc')}")

### 9. Package Model Artifacts to /kaggle/working/detection_results.zip

Package all trained models, checkpoints, and run cards into a downloadable ZIP archive.

In [ ]:
import os
import shutil
from pathlib import Path

models_dir = Path("/kaggle/working/models")
zip_output_path = "/kaggle/working/detection_results.zip"

if models_dir.exists():
    print(f"Archiving {models_dir} to {zip_output_path}...")
    shutil.make_archive(
        base_name="/kaggle/working/detection_results",
        format="zip",
        root_dir="/kaggle/working",
        base_dir="models",
    )
    zip_size_mb = os.path.getsize(zip_output_path) / (1024**2)
    print(f"[OK] Successfully created: {zip_output_path} ({zip_size_mb:.2f} MB)")
    print("    You can download this archive from the Kaggle Output sidebar after Save Version!")
else:
    print(f"[!] Warning: {models_dir} does not exist. Nothing to zip.")